In [5]:
import os, time
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, confusion_matrix
from xgboost import XGBClassifier

DATA_DIR = Path(os.environ.get("DATA_DIR", "/scratch/sm3664/medicare-fraud"))
PROC = DATA_DIR / "processed" / "part_b"
RES = DATA_DIR / "results" / "nb06_step1"
RES.mkdir(parents=True, exist_ok=True)

LABELED_FILE = PROC / "labeled" / "part_b_labeled_2019_2023.parquet"  # <-- set to your notebook-04 output
NPI, YEAR, LABEL = "Rndrng_NPI", "year", "fraud"           # <-- confirm label column name
PTYPE, ENT = "Rndrng_Prvdr_Type", "Rndrng_Prvdr_Ent_Cd"

SEED = 42
N_JOBS = 4          # 4 CPUs allocated, not the whole node
TEST_SIZE = 0.15
N_FOLDS = 10

In [6]:
for p in sorted(PROC.rglob("*.parquet")):
    print(p.relative_to(PROC), f"{p.stat().st_size/1e6:.0f} MB")

agg/part_b_agg_2019.parquet 150 MB
agg/part_b_agg_2020.parquet 149 MB
agg/part_b_agg_2021.parquet 154 MB
agg/part_b_agg_2022.parquet 159 MB
agg/part_b_agg_2023.parquet 161 MB
clean/part_b_clean_2019.parquet 199 MB
clean/part_b_clean_2020.parquet 193 MB
clean/part_b_clean_2021.parquet 195 MB
clean/part_b_clean_2022.parquet 201 MB
clean/part_b_clean_2023.parquet 193 MB
enriched/part_b_enriched_2019.parquet 262 MB
enriched/part_b_enriched_2020.parquet 260 MB
enriched/part_b_enriched_2021.parquet 268 MB
enriched/part_b_enriched_2022.parquet 276 MB
enriched/part_b_enriched_2023.parquet 280 MB
labeled/leie_cumulative_2019_2023.parquet 0 MB
labeled/part_b_labeled_2019_2023.parquet 1332 MB
part_b_agg_2019_2023.parquet 768 MB
part_b_enriched_2019_2023.parquet 1338 MB


In [7]:
df = pd.read_parquet(LABELED_FILE)
df[LABEL] = df[LABEL].astype(int)

print("shape:", df.shape)
print("fraud rows:", int(df[LABEL].sum()), " rate: %.4f%%" % (100 * df[LABEL].mean()))
print("unique NPIs:", df[NPI].nunique(), " fraud NPIs:", df.loc[df[LABEL] == 1, NPI].nunique())
print("NaN total:", int(df.isna().sum().sum()))
print(df.groupby(YEAR)[LABEL].agg(rows="size", fraud="sum"))

NUM_COLS = [c for c in df.columns if c not in {NPI, YEAR, LABEL, PTYPE, ENT}]
bad = [c for c in NUM_COLS if not pd.api.types.is_numeric_dtype(df[c])]
assert not bad, f"non-numeric feature columns: {bad}"
df[NUM_COLS] = df[NUM_COLS].astype("float32")
print(len(NUM_COLS), "numeric features")

shape: (5626423, 91)
fraud rows: 2652  rate: 0.0471%
unique NPIs: 1493778  fraud NPIs: 930
NaN total: 0
         rows  fraud
year                
2019  1093367    840
2020  1085313    671
2021  1123589    529
2022  1148873    369
2023  1175281    243
86 numeric features


In [8]:
prov = df.groupby(NPI)[LABEL].max().reset_index()
tr_npi, te_npi = train_test_split(prov[NPI], test_size=TEST_SIZE,
                                  stratify=prov[LABEL], random_state=SEED)

is_test = df[NPI].isin(te_npi)
train = df[~is_test].reset_index(drop=True)
test = df[is_test].reset_index(drop=True)

assert not set(train[NPI]) & set(test[NPI]), "NPI overlap between train and test"
te_npi.to_frame().to_parquet(RES / "test_npis.parquet")   # reuse this exact split later

for name, d in [("train", train), ("test", test)]:
    print(f"{name}: {d.shape}  fraud rows: {int(d[LABEL].sum())}  rate: {100*d[LABEL].mean():.4f}%")

train: (4782655, 91)  fraud rows: 2232  rate: 0.0467%
test: (843768, 91)  fraud rows: 420  rate: 0.0498%


In [9]:
prov_tr = train.groupby(NPI)[LABEL].max().reset_index()
prov_tr["fold"] = -1
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
for k, (_, idx) in enumerate(skf.split(prov_tr, prov_tr[LABEL])):
    prov_tr.loc[idx, "fold"] = k

fold_of = train[NPI].map(prov_tr.set_index(NPI)["fold"]).to_numpy()
folds = [(np.where(fold_of != k)[0], np.where(fold_of == k)[0]) for k in range(N_FOLDS)]

for k, (i_tr, i_va) in enumerate(folds):
    print(f"fold {k}: val rows {len(i_va):,}  val fraud {int(train[LABEL].iloc[i_va].sum())}")

fold 0: val rows 478,491  val fraud 230
fold 1: val rows 478,638  val fraud 233
fold 2: val rows 477,326  val fraud 239
fold 3: val rows 478,500  val fraud 206
fold 4: val rows 478,146  val fraud 213
fold 5: val rows 478,328  val fraud 231
fold 6: val rows 478,918  val fraud 210
fold 7: val rows 477,514  val fraud 228
fold 8: val rows 478,237  val fraud 229
fold 9: val rows 478,557  val fraud 213


In [10]:
FEATURE_SETS = {
    "base":           [],               # numeric only (old baseline)
    "base+ptype":     [PTYPE],
    "base+ptype+ent": [PTYPE, ENT],
}

def fit_encoder(d, cats):
    if not cats:
        return None
    enc = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50,
                        sparse_output=False, dtype=np.float32)
    enc.fit(d[cats].astype(str))        # fit on training rows only
    return enc

def make_X(d, cats, enc):
    X = d[NUM_COLS].to_numpy(dtype=np.float32)
    if enc is not None:
        X = np.hstack([X, enc.transform(d[cats].astype(str))])
    return X

def undersample(d, ratio, seed):
    pos = d[d[LABEL] == 1]
    neg = d[d[LABEL] == 0]
    neg = neg.sample(n=min(len(neg), int(ratio * len(pos))), random_state=seed)
    return pd.concat([pos, neg]).sample(frac=1, random_state=seed)

def make_model(name, seed):
    if name == "XGB":
        return XGBClassifier(max_depth=4, tree_method="hist", n_jobs=N_JOBS,
                             random_state=seed, eval_metric="auc")
    return RandomForestClassifier(max_depth=16, n_jobs=N_JOBS, random_state=seed)

def evaluate(y, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    tpr = tp / (tp + fn) if (tp + fn) else np.nan
    tnr = tn / (tn + fp) if (tn + fp) else np.nan
    return dict(AUC=roc_auc_score(y, p), AUPRC=average_precision_score(y, p),
                TPR=tpr, TNR=tnr, GMean=np.sqrt(tpr * tnr),
                TP=tp, FN=fn, FP=fp, TN=tn, thr=thr)

def run_one(tr, va, cats, model_name, ratio, seed):
    enc = fit_encoder(tr, cats)
    tr_s = undersample(tr, ratio, seed)
    Xtr, ytr = make_X(tr_s, cats, enc), tr_s[LABEL].to_numpy()
    model = make_model(model_name, seed).fit(Xtr, ytr)
    thr = ytr.mean()                    # prior of the training data (paper's rule): 1:1 -> 0.5
    p = model.predict_proba(make_X(va, cats, enc))[:, 1]
    return model, enc, p, evaluate(va[LABEL].to_numpy(), p, thr), Xtr.shape[1]

In [11]:
i_tr, i_va = folds[0]
tr0, va0 = train.iloc[i_tr], train.iloc[i_va]

for fs_name, cats in FEATURE_SETS.items():
    _, _, p, m, n_feat = run_one(tr0, va0, cats, "XGB", 1, SEED)
    print(f"\n{fs_name}: {n_feat} features | AUC {m['AUC']:.4f}  AUPRC {m['AUPRC']:.4f}  "
          f"TPR {m['TPR']:.3f}  TNR {m['TNR']:.3f}  thr {m['thr']:.3f}")
    print("predicted-positive rate: %.3f" % (p >= m["thr"]).mean())
    print(pd.DataFrame({"y": va0[LABEL].values, "p": p})
            .groupby("y")["p"].describe(percentiles=[.5, .9, .99]).round(3))


base: 86 features | AUC 0.7211  AUPRC 0.0025  TPR 0.630  TNR 0.736  thr 0.500
predicted-positive rate: 0.265
      count   mean    std    min    50%    90%    99%    max
y                                                           
0  478261.0  0.345  0.247  0.000  0.294  0.721  0.941  1.000
1     230.0  0.571  0.289  0.024  0.615  0.940  0.992  0.997

base+ptype: 188 features | AUC 0.7660  AUPRC 0.0038  TPR 0.652  TNR 0.749  thr 0.500
predicted-positive rate: 0.251
      count   mean    std    min    50%    90%    99%    max
y                                                           
0  478261.0  0.330  0.249  0.000  0.276  0.713  0.935  0.999
1     230.0  0.603  0.276  0.024  0.646  0.949  0.993  0.994

base+ptype+ent: 190 features | AUC 0.7613  AUPRC 0.0048  TPR 0.635  TNR 0.750  thr 0.500
predicted-positive rate: 0.250
      count   mean    std    min    50%    90%    99%    max
y                                                           
0  478261.0  0.326  0.248  0.000  0.274  0

In [12]:
SWEEP = [1, 5, 50, 100, 10**9]     # 10**9 = keep all negatives (paper's setup)
cats = [PTYPE]

for model_name in ["XGB"]:
    for ratio in SWEEP:
        t0 = time.time()
        _, _, p, m, _ = run_one(tr0, va0, cats, model_name, ratio, SEED)
        label = "none" if ratio == 10**9 else f"1:{ratio}"
        print(f"{model_name} {label:>6}  AUC {m['AUC']:.4f}  AUPRC {m['AUPRC']:.4f}  "
              f"TPR {m['TPR']:.3f}  TNR {m['TNR']:.3f}  thr {m['thr']:.5f}  ({time.time()-t0:.0f}s)")

XGB    1:1  AUC 0.7660  AUPRC 0.0038  TPR 0.652  TNR 0.749  thr 0.50000  (6s)
XGB    1:5  AUC 0.7525  AUPRC 0.0034  TPR 0.591  TNR 0.790  thr 0.16667  (6s)
XGB   1:50  AUC 0.7657  AUPRC 0.0035  TPR 0.526  TNR 0.801  thr 0.01961  (8s)
XGB  1:100  AUC 0.7635  AUPRC 0.0028  TPR 0.574  TNR 0.801  thr 0.00990  (10s)
XGB   none  AUC 0.7546  AUPRC 0.0032  TPR 0.565  TNR 0.796  thr 0.00047  (102s)


In [13]:
RATIOS = [1]          # add 3, 5, 50, 100 for the full grid; run that via sbatch
SEEDS = [SEED]        # add more seeds (e.g. [42, 7, 123]) once this runs end to end
MODELS = ["XGB", "RF"]

rows = []
for fs_name, cats in FEATURE_SETS.items():
    for model_name in MODELS:
        for ratio in RATIOS:
            for seed in SEEDS:
                for k, (i_tr, i_va) in enumerate(folds):
                    t0 = time.time()
                    _, _, _, m, _ = run_one(train.iloc[i_tr], train.iloc[i_va], cats, model_name, ratio, seed)
                    rows.append(dict(features=fs_name, model=model_name, ratio=ratio, seed=seed, fold=k, **m))
                    print(f"{fs_name:15s} {model_name} 1:{ratio} s{seed} f{k}  "
                          f"AUC {m['AUC']:.4f}  AUPRC {m['AUPRC']:.4f}  ({time.time()-t0:.0f}s)")

cv = pd.DataFrame(rows)
cv.to_csv(RES / "cv_results.csv", index=False)

base            XGB 1:1 s42 f0  AUC 0.7211  AUPRC 0.0025  (3s)
base            XGB 1:1 s42 f1  AUC 0.7577  AUPRC 0.0103  (3s)
base            XGB 1:1 s42 f2  AUC 0.7285  AUPRC 0.0018  (3s)
base            XGB 1:1 s42 f3  AUC 0.7120  AUPRC 0.0024  (3s)
base            XGB 1:1 s42 f4  AUC 0.7864  AUPRC 0.0037  (3s)
base            XGB 1:1 s42 f5  AUC 0.6634  AUPRC 0.0014  (3s)
base            XGB 1:1 s42 f6  AUC 0.7481  AUPRC 0.0026  (3s)
base            XGB 1:1 s42 f7  AUC 0.7606  AUPRC 0.0036  (3s)
base            XGB 1:1 s42 f8  AUC 0.7029  AUPRC 0.0020  (3s)
base            XGB 1:1 s42 f9  AUC 0.7141  AUPRC 0.0021  (3s)
base            RF 1:1 s42 f0  AUC 0.7267  AUPRC 0.0051  (5s)
base            RF 1:1 s42 f1  AUC 0.7844  AUPRC 0.0042  (5s)
base            RF 1:1 s42 f2  AUC 0.7567  AUPRC 0.0025  (5s)
base            RF 1:1 s42 f3  AUC 0.7240  AUPRC 0.0014  (5s)
base            RF 1:1 s42 f4  AUC 0.7968  AUPRC 0.0036  (5s)
base            RF 1:1 s42 f5  AUC 0.6650  AUPRC 0.0034  (5s

In [14]:
def ci95(x):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    h = 1.96 * x.std(ddof=1) / np.sqrt(len(x))
    return f"{x.mean():.4f} ({x.mean()-h:.4f}, {x.mean()+h:.4f})"

summary = cv.groupby(["features", "model", "ratio"]).agg(
    AUC=("AUC", ci95), AUPRC=("AUPRC", ci95), TPR=("TPR", ci95),
    TNR=("TNR", ci95), GMean=("GMean", ci95),
    FN_per_fold=("FN", "mean"), FP_per_fold=("FP", "mean"))
summary.to_csv(RES / "cv_summary.csv")
summary

AUC                    AUPRC  \
features       model ratio                                                     
base           RF    1      0.7399 (0.7162, 0.7636)  0.0030 (0.0022, 0.0037)   
               XGB   1      0.7295 (0.7078, 0.7512)  0.0032 (0.0016, 0.0048)   
base+ptype     RF    1      0.7542 (0.7306, 0.7778)  0.0036 (0.0017, 0.0055)   
               XGB   1      0.7647 (0.7421, 0.7874)  0.0033 (0.0025, 0.0040)   
base+ptype+ent RF    1      0.7575 (0.7363, 0.7786)  0.0036 (0.0025, 0.0048)   
               XGB   1      0.7659 (0.7461, 0.7857)  0.0034 (0.0025, 0.0043)   

                                                TPR                      TNR  \
features       model ratio                                                     
base           RF    1      0.5827 (0.5320, 0.6335)  0.7630 (0.7582, 0.7677)   
               XGB   1      0.5975 (0.5523, 0.6426)  0.7320 (0.7285, 0.7356)   
base+ptype     RF    1      0.6113 (0.5711, 0.6514)  0.7702 (0.7660, 0.7744)   
               XGB   1      0.6364 (0.5884, 0.6843)  0.7454 (0.7416, 0.7493)   
base+ptype+ent RF    1      0.6057 (0.5611, 0.6504)  0.7722 (0.7685, 0.7760)   
               XGB   1      0.6554 (0.6142, 0.6966)  0.7442 (0.7405, 0.7479)   

                                              GMean  FN_per_fold  FP_per_fold  
features       model ratio                                                     
base           RF    1      0.6651 (0.6367, 0.6934)         93.2     113297.8  
               XGB   1      0.6601 (0.6353, 0.6848)         89.9     128109.8  
base+ptype     RF    1      0.6852 (0.6636, 0.7068)         86.8     109851.3  
               XGB   1      0.6875 (0.6628, 0.7122)         81.4     121686.9  
base+ptype+ent RF    1      0.6828 (0.6584, 0.7072)         88.0     108876.8  
               XGB   1      0.6975 (0.6767, 0.7182)         77.1     122286.2

In [15]:
_, _, p, m, _ = run_one(tr0, va0, [PTYPE], "XGB", 1, SEED)
v = pd.DataFrame({YEAR: va0[YEAR].values, "y": va0[LABEL].values, "p": p})
for yr, g in v.groupby(YEAR):
    print(yr, f"fraud {int(g.y.sum()):4d}  AUC {roc_auc_score(g.y, g.p):.4f}")

2019 fraud   69  AUC 0.7866
2020 fraud   59  AUC 0.7648
2021 fraud   45  AUC 0.7211
2022 fraud   34  AUC 0.7479
2023 fraud   23  AUC 0.7568
